# KM Curves for Overlap Markers: Adjusted vs Unadjusted

For markers that are significant across multiple specifications (identified by `marker_overlap_analysis.ipynb`), visualize Kaplan-Meier curves:

**Track 1 (ICI-only)**:
- Two panels per marker: ATE-weighted (generalizability) vs Unweighted
- Side-by-side for cohort1 (unmatched) vs cohort2 (matched) when both are available

**Track 2 (Interaction)**:
- Four panels per marker: ICI vs Non-ICI arms x ATE-weighted vs noIPTW
- Side-by-side for cohort1 vs cohort2 when both are available

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.exceptions import StatisticalWarning
from sklearn.linear_model import LogisticRegression

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

COLOR_POS = "#D62728"   # red for marker+
COLOR_NEG = "#1F77B4"   # blue for marker-
CI_ALPHA = 0.18

# ── Paths ─────────────────────────────────────────────────────────────
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
MARKER_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/')
COMPILED_DIR = os.path.join(MARKER_PATH, 'compiled_results/')
FIGURE_PATH = '/data/gusev/USERS/jpconnor/figures/clinical_text_embedding_project/'
KM_FIG_PATH = os.path.join(FIGURE_PATH, 'biomarker_analysis/overlap_KM/')
os.makedirs(KM_FIG_PATH, exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────
COHORTS = ['cohort1', 'cohort2']
PS_MODELS = ['covariates_only', 'covariates_plus_embeddings']
N_BOOT = 250
GRID_POINTS = 200
SEED = 42
MAX_TIME_DAYS = None  # set e.g. 1825 to cap at 5 years

# ── Load overlap markers ──────────────────────────────────────────────
overlap_markers = pd.read_csv(os.path.join(COMPILED_DIR, 'overlap_markers_for_km.csv'))
print(f"Loaded {len(overlap_markers)} (marker, cancer_type) pairs for KM analysis")
display(overlap_markers)

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────

COMMON_SUPPORT_PCT = (0.5, 99.5)
IPTW_TRUNC_PCT = (1, 99)
EPS = 1e-6


def recalibrate_propensity_within_subset(df, ps_col="ICI_prediction", treat_col="PX_on_ICI"):
    """Recalibrate propensity scores within a cancer-type subset."""
    ps = df[[ps_col]].values
    y = df[treat_col].values.astype(int)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(ps, y)
    return lr.predict_proba(ps)[:, 1]


def prepare_weighted_df(full_df, cancer_type):
    """Subset to cancer type, trim common support, compute stabilized ATE weights.

    Mirrors run_IPTW_analysis.py logic:
    - Recalibrate PS within cancer-type subset
    - Common support trimming at (0.5, 99.5) percentiles
    - Stabilized ATE weights: w = p_treated/ps (treated), (1-p_treated)/(1-ps) (control)
    - Truncation at (1, 99) percentiles
    - Generalizability weights for ICI-only: w = 1/ps (Track 1)
    """
    if cancer_type == "pan_cancer":
        df = full_df.copy()
    else:
        ct_col = f"CANCER_TYPE_{cancer_type}"
        if ct_col not in full_df.columns:
            raise ValueError(f"Column {ct_col} not found")
        df = full_df.loc[full_df[ct_col].astype(bool)].copy()
        if df["PX_on_ICI"].nunique() >= 2 and len(df) > 10:
            df["ICI_prediction"] = recalibrate_propensity_within_subset(df)

    ps_raw = df["ICI_prediction"].clip(EPS, 1 - EPS)
    ps_t = ps_raw[df["PX_on_ICI"] == 1]
    ps_c = ps_raw[df["PX_on_ICI"] == 0]

    lower = max(np.percentile(ps_t, COMMON_SUPPORT_PCT[0]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[0]))
    upper = min(np.percentile(ps_t, COMMON_SUPPORT_PCT[1]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[1]))
    df = df[(ps_raw >= lower) & (ps_raw <= upper)].copy()

    ps = df["ICI_prediction"].clip(EPS, 1 - EPS)
    treat = df["PX_on_ICI"]
    p_treated = treat.mean()

    # Stabilized ATE weights (Track 2)
    w_ate = np.where(treat == 1, p_treated / ps, (1 - p_treated) / (1 - ps))
    lo, hi = np.percentile(w_ate, IPTW_TRUNC_PCT)
    df["IPTW_ATE"] = np.clip(w_ate, lo, hi)

    # Generalizability weights for ICI-only (Track 1): w = 1/ps
    ici_mask = treat == 1
    ici_ps = ps[ici_mask]
    w_gen = 1.0 / ici_ps
    lo_gen, hi_gen = np.percentile(w_gen, IPTW_TRUNC_PCT)
    if np.isfinite(lo_gen) and np.isfinite(hi_gen):
        df.loc[ici_mask, "IPTW_GEN"] = np.clip(w_gen, lo_gen, hi_gen).values
    else:
        df.loc[ici_mask, "IPTW_GEN"] = 1.0

    return df


def bootstrap_weighted_km(durations, events, weights, grid,
                          n_boot=250, seed=42, renormalize=True):
    """Bootstrap weighted KM -> median, lo, hi survival curves on grid."""
    n = len(durations)
    if n == 0:
        return np.ones(len(grid)), np.ones(len(grid)), np.ones(len(grid))

    dur = np.asarray(durations)
    evt = np.asarray(events)
    wgt = np.asarray(weights, dtype=float)

    if renormalize:
        s = wgt.sum()
        if s > 0:
            wgt = wgt * (n / s)

    rng = np.random.default_rng(seed)
    km = KaplanMeierFitter()
    surv_mat = np.empty((n_boot, len(grid)))

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=StatisticalWarning)
            km.fit(dur[idx], evt[idx], weights=wgt[idx])
        surv_mat[b, :] = km.survival_function_at_times(grid).values

    return (np.median(surv_mat, axis=0),
            np.quantile(surv_mat, 0.025, axis=0),
            np.quantile(surv_mat, 0.975, axis=0))


def unweighted_km(durations, events, grid):
    """Standard KM + Greenwood CI interpolated onto grid."""
    km = KaplanMeierFitter()
    km.fit(durations, events)
    surv = km.survival_function_at_times(grid).values
    ci = km.confidence_interval_survival_function_
    ci_lo = np.interp(grid, ci.index, ci.iloc[:, 0])
    ci_hi = np.interp(grid, ci.index, ci.iloc[:, 1])
    return surv, ci_lo, ci_hi


def _format_ax(ax, title, xlabel="Time (days)", ylabel="Survival probability"):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.02, 1.02)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _p_str(p):
    return f"p={p:.2e}" if p < 0.001 else f"p={p:.3f}"


print("Helper functions defined.")

In [ ]:
# ── Load IPTW DataFrames and prepare weighted data ────────────────────

needed_types = set(overlap_markers['cancer_type'].unique())

# weighted_cache[(cohort, ps_model, cancer_type)] -> weighted DataFrame
weighted_cache = {}

for cohort in COHORTS:
    for ps_model in PS_MODELS:
        spec_label = f"{cohort}_{ps_model}"
        iptw_file = os.path.join(MARKER_PATH, f"IPTW_df_{spec_label}.csv")
        print(f"\n{'=' * 60}")
        print(f"Loading {spec_label}: {iptw_file}")

        try:
            spec_df = pd.read_csv(iptw_file)
        except FileNotFoundError:
            print(f"  FILE NOT FOUND — skipping")
            continue

        n_ici = int((spec_df["PX_on_ICI"] == 1).sum())
        n_ctrl = int((spec_df["PX_on_ICI"] == 0).sum())
        print(f"  {len(spec_df)} rows ({n_ici} ICI, {n_ctrl} non-ICI)")

        for ct in sorted(needed_types):
            print(f"  Preparing {ct}...", end=" ")
            try:
                wdf = prepare_weighted_df(spec_df, ct)
                weighted_cache[(cohort, ps_model, ct)] = wdf
                n = len(wdf)
                n_i = int((wdf["PX_on_ICI"] == 1).sum())
                print(f"{n} patients ({n_i} ICI, {n - n_i} non-ICI)")
            except Exception as e:
                print(f"ERROR: {e}")

print(f"\nCached {len(weighted_cache)} (cohort, ps_model, cancer_type) combinations")

## Track 1: ATE-Weighted vs Unweighted KM (ICI-only)

For each overlap marker, plot KM curves in ICI patients:
- **Left**: ATE generalizability-weighted (bootstrap CI)
- **Right**: Unweighted (Greenwood CI)

When the marker overlaps across cohorts, plot cohort1 and cohort2 side-by-side (4 panels).

In [ ]:
# ── Track 1: ATE vs Unweighted KM, ICI-only ──────────────────────────

def _plot_t1_panel(ax, ici_df, marker, grid, weighted, weight_col="IPTW_GEN"):
    """Plot a single Track 1 KM panel (weighted or unweighted) on ax."""
    d = ici_df[["tt_death", "death", marker, weight_col]].dropna().copy()
    d[marker] = (pd.to_numeric(d[marker], errors="coerce").fillna(0) > 0).astype(int)
    d_pos = d[d[marker] == 1]
    d_neg = d[d[marker] == 0]

    if len(d_pos) < 5 or len(d_neg) < 5:
        ax.text(0.5, 0.5, f"n+ = {len(d_pos)}, n- = {len(d_neg)}\n(insufficient)",
                ha='center', va='center', transform=ax.transAxes)
        return None

    if weighted:
        # Weighted log-rank
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"],
            weights_A=d_pos[weight_col], weights_B=d_neg[weight_col])

        med_p, lo_p, hi_p = bootstrap_weighted_km(
            d_pos["tt_death"], d_pos["death"], d_pos[weight_col], grid, N_BOOT, SEED)
        med_n, lo_n, hi_n = bootstrap_weighted_km(
            d_neg["tt_death"], d_neg["death"], d_neg[weight_col], grid, N_BOOT, SEED + 1)

        ax.plot(grid, med_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, med_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)
        _format_ax(ax, f"ATE-weighted\n{_p_str(lr.p_value)}")
    else:
        # Unweighted log-rank
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"])

        s_p, lo_p, hi_p = unweighted_km(d_pos["tt_death"], d_pos["death"], grid)
        s_n, lo_n, hi_n = unweighted_km(d_neg["tt_death"], d_neg["death"], grid)

        ax.plot(grid, s_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, s_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)
        _format_ax(ax, f"Unweighted\n{_p_str(lr.p_value)}")

    ax.legend(fontsize=9, loc="lower left")
    return lr.p_value


def plot_track1_overlap_km(marker, cancer_type, ps_model):
    """Plot Track 1 KM for a marker across cohorts (2x2 or 1x2)."""
    available_cohorts = [
        c for c in COHORTS
        if (c, ps_model, cancer_type) in weighted_cache
    ]
    if not available_cohorts:
        print(f"  Skip {marker}/{cancer_type}/{ps_model}: no cached data")
        return

    n_cohorts = len(available_cohorts)
    fig, axes = plt.subplots(n_cohorts, 2, figsize=(14, 5.5 * n_cohorts),
                             squeeze=False, sharey=True)

    for row, cohort in enumerate(available_cohorts):
        df = weighted_cache[(cohort, ps_model, cancer_type)]
        ici_df = df[df["PX_on_ICI"] == 1].copy()

        tmax = float(ici_df["tt_death"].max()) if len(ici_df) else 1.0
        if MAX_TIME_DAYS is not None:
            tmax = min(tmax, MAX_TIME_DAYS)
        grid = np.linspace(0, tmax, GRID_POINTS)

        _plot_t1_panel(axes[row, 0], ici_df, marker, grid, weighted=True)
        _plot_t1_panel(axes[row, 1], ici_df, marker, grid, weighted=False)

        # Row label
        cohort_label = "Unmatched" if cohort == "cohort1" else "Matched (1:1)"
        axes[row, 0].set_ylabel(f"{cohort_label}\nSurvival probability")

    fig.suptitle(f"{marker} — {cancer_type} — {ps_model}\nTrack 1: ICI-only",
                 fontweight="bold", y=1.02)
    fig.tight_layout()

    fname = f"T1_{marker}_{cancer_type}_{ps_model}.png"
    fig.savefig(os.path.join(KM_FIG_PATH, fname))
    plt.show()


# Run for all overlap markers, both PS models
for _, row in overlap_markers.iterrows():
    marker, cancer_type = row['marker'], row['cancer_type']
    for ps_model in PS_MODELS:
        print(f"\n--- {marker} / {cancer_type} / {ps_model} ---")
        plot_track1_overlap_km(marker, cancer_type, ps_model)

## Track 2: ATE-Weighted vs noIPTW KM (ICI vs Non-ICI)

For each overlap marker, plot KM curves split by treatment arm and weighting:
- **Columns**: ATE-weighted (left) vs noIPTW (right)
- **Rows**: ICI arm (top) vs Non-ICI arm (bottom)

When available across cohorts, both are shown.

In [ ]:
# ── Track 2: ATE vs noIPTW KM, ICI vs Non-ICI ────────────────────────

def _plot_t2_arm_panel(ax, arm_df, marker, grid, weighted, weight_col="IPTW_ATE"):
    """Plot KM for one treatment arm (ICI or Non-ICI), weighted or not."""
    cols = ["tt_death", "death", marker]
    if weighted:
        cols.append(weight_col)
    d = arm_df[cols].dropna().copy()
    d[marker] = (pd.to_numeric(d[marker], errors="coerce").fillna(0) > 0).astype(int)
    d_pos = d[d[marker] == 1]
    d_neg = d[d[marker] == 0]

    if len(d_pos) < 5 or len(d_neg) < 5:
        ax.text(0.5, 0.5, f"n+ = {len(d_pos)}, n- = {len(d_neg)}\n(insufficient)",
                ha='center', va='center', transform=ax.transAxes)
        return None

    if weighted:
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"],
            weights_A=d_pos[weight_col], weights_B=d_neg[weight_col])

        med_p, lo_p, hi_p = bootstrap_weighted_km(
            d_pos["tt_death"], d_pos["death"], d_pos[weight_col], grid, N_BOOT, SEED)
        med_n, lo_n, hi_n = bootstrap_weighted_km(
            d_neg["tt_death"], d_neg["death"], d_neg[weight_col], grid, N_BOOT, SEED + 1)

        ax.plot(grid, med_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, med_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)
    else:
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"])

        s_p, lo_p, hi_p = unweighted_km(d_pos["tt_death"], d_pos["death"], grid)
        s_n, lo_n, hi_n = unweighted_km(d_neg["tt_death"], d_neg["death"], grid)

        ax.plot(grid, s_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, s_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)

    ax.legend(fontsize=8, loc="lower left")
    return lr.p_value


def plot_track2_overlap_km(marker, cancer_type, ps_model):
    """Plot Track 2 KM: rows = cohorts, cols = (ATE ICI, ATE nonICI, noIPTW ICI, noIPTW nonICI)."""
    available_cohorts = [
        c for c in COHORTS
        if (c, ps_model, cancer_type) in weighted_cache
    ]
    if not available_cohorts:
        print(f"  Skip {marker}/{cancer_type}/{ps_model}: no cached data")
        return

    n_cohorts = len(available_cohorts)
    # 4 columns: ATE-ICI, ATE-nonICI, noIPTW-ICI, noIPTW-nonICI
    fig, axes = plt.subplots(n_cohorts, 4, figsize=(24, 5.5 * n_cohorts),
                             squeeze=False, sharey=True)

    for row, cohort in enumerate(available_cohorts):
        df = weighted_cache[(cohort, ps_model, cancer_type)]
        ici_df = df[df["PX_on_ICI"] == 1].copy()
        nonici_df = df[df["PX_on_ICI"] == 0].copy()

        tmax = float(df["tt_death"].max()) if len(df) else 1.0
        if MAX_TIME_DAYS is not None:
            tmax = min(tmax, MAX_TIME_DAYS)
        grid = np.linspace(0, tmax, GRID_POINTS)

        # ATE-weighted ICI
        p_val = _plot_t2_arm_panel(axes[row, 0], ici_df, marker, grid,
                                   weighted=True, weight_col="IPTW_ATE")
        _format_ax(axes[row, 0], f"ICI — ATE-weighted\n{_p_str(p_val) if p_val else ''}")

        # ATE-weighted non-ICI
        p_val = _plot_t2_arm_panel(axes[row, 1], nonici_df, marker, grid,
                                   weighted=True, weight_col="IPTW_ATE")
        _format_ax(axes[row, 1], f"Non-ICI — ATE-weighted\n{_p_str(p_val) if p_val else ''}")

        # noIPTW ICI
        p_val = _plot_t2_arm_panel(axes[row, 2], ici_df, marker, grid, weighted=False)
        _format_ax(axes[row, 2], f"ICI — noIPTW\n{_p_str(p_val) if p_val else ''}")

        # noIPTW non-ICI
        p_val = _plot_t2_arm_panel(axes[row, 3], nonici_df, marker, grid, weighted=False)
        _format_ax(axes[row, 3], f"Non-ICI — noIPTW\n{_p_str(p_val) if p_val else ''}")

        # Row label
        cohort_label = "Unmatched" if cohort == "cohort1" else "Matched (1:1)"
        axes[row, 0].set_ylabel(f"{cohort_label}\nSurvival probability")

    fig.suptitle(f"{marker} — {cancer_type} — {ps_model}\n"
                 f"Track 2: ICI vs Non-ICI x ATE vs noIPTW",
                 fontweight="bold", y=1.02)
    fig.tight_layout()

    fname = f"T2_{marker}_{cancer_type}_{ps_model}.png"
    fig.savefig(os.path.join(KM_FIG_PATH, fname))
    plt.show()


# Run for all overlap markers
for _, row in overlap_markers.iterrows():
    marker, cancer_type = row['marker'], row['cancer_type']
    for ps_model in PS_MODELS:
        print(f"\n--- {marker} / {cancer_type} / {ps_model} ---")
        plot_track2_overlap_km(marker, cancer_type, ps_model)

## Summary: KM p-value comparison across weighting schemes

Collect weighted vs unweighted log-rank p-values for all overlap markers to see how adjustment shifts significance.

In [ ]:
# ── Collect log-rank p-values: weighted vs unweighted ─────────────────

pval_rows = []

for _, mrow in overlap_markers.iterrows():
    marker, cancer_type = mrow['marker'], mrow['cancer_type']
    for cohort in COHORTS:
        for ps_model in PS_MODELS:
            key = (cohort, ps_model, cancer_type)
            if key not in weighted_cache:
                continue

            df = weighted_cache[key]

            # Track 1: ICI-only
            ici_df = df[df["PX_on_ICI"] == 1].copy()
            d = ici_df[["tt_death", "death", marker, "IPTW_GEN"]].dropna().copy()
            d[marker] = (pd.to_numeric(d[marker], errors="coerce").fillna(0) > 0).astype(int)
            d_pos, d_neg = d[d[marker] == 1], d[d[marker] == 0]

            if len(d_pos) >= 5 and len(d_neg) >= 5:
                lr_w = logrank_test(
                    d_pos["tt_death"], d_neg["tt_death"],
                    event_observed_A=d_pos["death"], event_observed_B=d_neg["death"],
                    weights_A=d_pos["IPTW_GEN"], weights_B=d_neg["IPTW_GEN"])
                lr_u = logrank_test(
                    d_pos["tt_death"], d_neg["tt_death"],
                    event_observed_A=d_pos["death"], event_observed_B=d_neg["death"])

                pval_rows.append({
                    'marker': marker, 'cancer_type': cancer_type,
                    'cohort': cohort, 'ps_model': ps_model,
                    'track': 1,
                    'p_weighted': lr_w.p_value,
                    'p_unweighted': lr_u.p_value,
                    'n_pos': len(d_pos), 'n_neg': len(d_neg),
                })

            # Track 2: full cohort (ICI arm only for simplicity)
            d2 = ici_df[["tt_death", "death", marker, "IPTW_ATE"]].dropna().copy()
            d2[marker] = (pd.to_numeric(d2[marker], errors="coerce").fillna(0) > 0).astype(int)
            d2_pos, d2_neg = d2[d2[marker] == 1], d2[d2[marker] == 0]

            if len(d2_pos) >= 5 and len(d2_neg) >= 5:
                lr_w2 = logrank_test(
                    d2_pos["tt_death"], d2_neg["tt_death"],
                    event_observed_A=d2_pos["death"], event_observed_B=d2_neg["death"],
                    weights_A=d2_pos["IPTW_ATE"], weights_B=d2_neg["IPTW_ATE"])
                lr_u2 = logrank_test(
                    d2_pos["tt_death"], d2_neg["tt_death"],
                    event_observed_A=d2_pos["death"], event_observed_B=d2_neg["death"])

                pval_rows.append({
                    'marker': marker, 'cancer_type': cancer_type,
                    'cohort': cohort, 'ps_model': ps_model,
                    'track': 2,
                    'p_weighted': lr_w2.p_value,
                    'p_unweighted': lr_u2.p_value,
                    'n_pos': len(d2_pos), 'n_neg': len(d2_neg),
                })

pval_df = pd.DataFrame(pval_rows)
print(f"Collected {len(pval_df)} p-value comparisons")

if len(pval_df) > 0:
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(-np.log10(pval_df['p_unweighted']),
               -np.log10(pval_df['p_weighted']),
               c=pval_df['track'].map({1: '#1f77b4', 2: '#d62728'}),
               s=30, alpha=0.6, edgecolors='k', linewidth=0.3)

    lims = [0, max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=1)
    ax.axhline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)
    ax.axvline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)

    ax.set_xlabel('-log10(p) Unweighted')
    ax.set_ylabel('-log10(p) Weighted')
    ax.set_title('Log-rank p-values: Weighted vs Unweighted')
    ax.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', label='Track 1'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', label='Track 2'),
    ], loc='upper left')

    fig.tight_layout()
    fig.savefig(os.path.join(KM_FIG_PATH, 'pvalue_weighted_vs_unweighted.png'))
    plt.show()

    pval_df.to_csv(os.path.join(KM_FIG_PATH, 'km_pvalue_comparison.csv'), index=False)
    print(f"Saved to {os.path.join(KM_FIG_PATH, 'km_pvalue_comparison.csv')}")